# Phase 2 — Step 5: Consolidated Metrics & Analysis

## Objective
Consolidate all empirical data from Steps 1–4 into comparative tables across four dimensions:
1. **Retrieval Quality**: MRR, Relevant@3, Relevant@5
2. **Security**: Breach count, Breach rate, Security PASS rate
3. **Latency**: Average execution time (ms) per pipeline
4. **Context Restoration**: Parent-child reconstruction effectiveness

Generate the final cross-step comparison that will feed directly into the TFG report (Section 6.2.9).

In [1]:
# Cell 1 — Imports & Load all results
import json, os
import pandas as pd
import numpy as np

BASE = os.path.dirname(os.path.abspath("__file__"))
RES = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph2")

# Step 1: Baseline
s1_engines = pd.read_csv(os.path.join(RES, "ph2_baseline_engine_summary.csv"))
s1_metrics = pd.read_csv(os.path.join(RES, "ph2_baseline_metrics.csv"))

# Step 2: Dynamic RRF
s2_engines = pd.read_csv(os.path.join(RES, "ph2_step2_engine_summary.csv"))
s2_delta = pd.read_csv(os.path.join(RES, "ph2_step2_delta.csv"))
s2_comparison = pd.read_csv(os.path.join(RES, "ph2_step2_comparison.csv"))

# Step 3: Parent-Child
s3_results = pd.read_csv(os.path.join(RES, "ph2_step3_reconstruction_results.csv"))
with open(os.path.join(RES, "ph2_step3_reconstruction_detail.json"), "r", encoding="utf-8") as f:
    s3_detail = json.load(f)

# Step 4: Asymmetric Fusion
s4_results = pd.read_csv(os.path.join(RES, "ph2_step4_asymmetric_results.csv"))
with open(os.path.join(RES, "ph2_step4_asymmetric_detail.json"), "r", encoding="utf-8") as f:
    s4_detail = json.load(f)

print("All results loaded.")
print(f"  Step 1: {len(s1_metrics)} query-engine rows, {len(s1_engines)} engine summaries")
print(f"  Step 2: {len(s2_comparison)} comparison rows, {len(s2_delta)} delta rows")
print(f"  Step 3: {len(s3_results)} test cases")
print(f"  Step 4: {len(s4_results)} test cases")

All results loaded.
  Step 1: 24 query-engine rows, 3 engine summaries
  Step 2: 16 comparison rows, 14 delta rows
  Step 3: 8 test cases
  Step 4: 12 test cases


## Table 1 — Cross-Step Evolution: Retrieval Quality & Security

This master table compares every pipeline configuration across all 4 steps on the same metrics.

**Note on comparability:** Steps 1-2 use 8/14 queries respectively (same corpus, different test sets). Steps 3-4 use 8/12 queries with different user profiles. The table presents each step's metrics on its own test set, since the purpose is to show the *evolution of the system*, not direct numerical comparison.

In [2]:
# Cell 2 — Master comparison table

# Step 1 baseline (8 queries, ensemble_rrf only)
s1_ens = s1_metrics[s1_metrics["engine"] == "ensemble_rrf"]
s1_breaches = s1_ens["breaches"].sum()
s1_queries_fail = (s1_ens["security"] == "FAIL").sum()
s1_total_q = len(s1_ens)
s1_lat = s1_ens["latency_ms"].mean()

# Step 2 (14 queries, baseline vs dynamic)
s2_base = s2_comparison[s2_comparison["mode"] == "baseline"]
s2_dyn = s2_comparison[s2_comparison["mode"] == "dynamic"]

# Step 3 (8 queries)
s3_before_breaches = s3_results["before_breaches"].sum()
s3_after_breaches = s3_results["after_breaches"].sum()
s3_before_fail = (s3_results["before_sec"] == "FAIL").sum()
s3_after_fail = (s3_results["after_sec"] == "FAIL").sum()

# Step 4 (12 queries)
s4_unsec_breaches = s4_results["unsec_breaches"].sum()
s4_sec_breaches = s4_results["sec_breaches"].sum()
s4_unsec_fail = (s4_results["unsec_security"] == "FAIL").sum()
s4_sec_fail = (s4_results["sec_security"] == "FAIL").sum()

# Build master table
master = pd.DataFrame([
    {
        "Step": "1 — Baseline",
        "Pipeline": "Ensemble RRF (α=0.5)",
        "Test Queries": s1_total_q,
        "MRR": round(s1_ens["breaches"].count() and s2_engines[s2_engines["Engine"]=="baseline_0.5"]["Avg_MRR"].values[0], 4),
        "R@3": int(s2_engines[s2_engines["Engine"]=="baseline_0.5"]["Tot_R@3"].values[0]),
        "R@5": int(s2_engines[s2_engines["Engine"]=="baseline_0.5"]["Tot_R@5"].values[0]),
        "Total Breaches": int(s2_engines[s2_engines["Engine"]=="baseline_0.5"]["Tot_Breaches"].values[0]),
        "Security PASS": f"0/{14}",
        "Avg Latency (ms)": round(s2_engines[s2_engines["Engine"]=="baseline_0.5"]["Avg_Lat"].values[0], 1),
    },
    {
        "Step": "2 — Dynamic RRF",
        "Pipeline": "Intent Router + α dynamic",
        "Test Queries": 14,
        "MRR": round(s2_engines[s2_engines["Engine"]=="dynamic"]["Avg_MRR"].values[0], 4),
        "R@3": int(s2_engines[s2_engines["Engine"]=="dynamic"]["Tot_R@3"].values[0]),
        "R@5": int(s2_engines[s2_engines["Engine"]=="dynamic"]["Tot_R@5"].values[0]),
        "Total Breaches": int(s2_engines[s2_engines["Engine"]=="dynamic"]["Tot_Breaches"].values[0]),
        "Security PASS": f"0/{14}",
        "Avg Latency (ms)": round(s2_engines[s2_engines["Engine"]=="dynamic"]["Avg_Lat"].values[0], 1),
    },
    {
        "Step": "3 — Parent-Child",
        "Pipeline": "Reconstruction + RBAC gate",
        "Test Queries": len(s3_results),
        "MRR": "N/A",
        "R@3": "N/A",
        "R@5": "N/A",
        "Total Breaches": f"{s3_before_breaches} → {s3_after_breaches}",
        "Security PASS": f"{len(s3_results) - s3_after_fail}/{len(s3_results)}",
        "Avg Latency (ms)": f"{s3_results['lat_retrieval_ms'].mean():.1f} + <0.1",
    },
    {
        "Step": "4 — Asymmetric Fusion",
        "Pipeline": "Pre+Post filter + Secure RRF",
        "Test Queries": len(s4_results),
        "MRR": f"{s4_results['unsec_mrr'].mean():.3f} → {s4_results['sec_mrr'].mean():.3f}",
        "R@3": "N/A",
        "R@5": "N/A",
        "Total Breaches": f"{s4_unsec_breaches} → {s4_sec_breaches}",
        "Security PASS": f"{len(s4_results) - s4_sec_fail}/{len(s4_results)}",
        "Avg Latency (ms)": f"{s4_results['unsec_lat_ms'].mean():.1f} → {s4_results['sec_lat_ms'].mean():.1f}",
    },
])

print("=" * 110)
print("  TABLE 1: CROSS-STEP EVOLUTION — RETRIEVAL QUALITY & SECURITY")
print("=" * 110)
print(master.to_string(index=False))
print()

  TABLE 1: CROSS-STEP EVOLUTION — RETRIEVAL QUALITY & SECURITY
                 Step                     Pipeline  Test Queries           MRR R@3 R@5 Total Breaches Security PASS Avg Latency (ms)
         1 — Baseline         Ensemble RRF (α=0.5)             8        0.8929  28  32             47          0/14              8.1
      2 — Dynamic RRF    Intent Router + α dynamic            14        0.8214  27  33             46          0/14              8.3
     3 — Parent-Child   Reconstruction + RBAC gate             8           N/A N/A N/A        19 → 14           2/8       8.3 + <0.1
4 — Asymmetric Fusion Pre+Post filter + Secure RRF            12 0.944 → 0.417 N/A N/A         38 → 0         12/12       9.0 → 10.0



In [3]:
# Cell 3 — Table 2: Security evolution across steps

print("=" * 90)
print("  TABLE 2: SECURITY BREACH EVOLUTION")
print("=" * 90)

# Step 1 per-engine breaches
print("\n── Step 1: Baseline breach rates (8 queries) ──")
print(s1_engines[["Engine", "Total Queries", "Total Breaches", "Breach Rate", "Security"]].to_string(index=False))

# Step 2: breaches per engine (14 queries)
print("\n── Step 2: Engine comparison (14 queries) ──")
print(s2_engines[["Engine", "Tot_Breaches", "Queries_FAIL"]].to_string(index=False))

# Step 3: before/after
print("\n── Step 3: Parent-Child reconstruction effect (8 tests) ──")
s3_summary = s3_results[["tid", "description", "before_breaches", "after_breaches",
                          "merges", "drops", "before_sec", "after_sec", "match"]]
print(s3_summary.to_string(index=False))
print(f"\n  Totals: breaches {s3_before_breaches} → {s3_after_breaches} "
      f"(Δ={s3_after_breaches - s3_before_breaches}), "
      f"PASS rate: {len(s3_results) - s3_before_fail}/{len(s3_results)} → "
      f"{len(s3_results) - s3_after_fail}/{len(s3_results)}")

# Step 4: unsecured vs secure
print("\n── Step 4: Asymmetric Fusion (12 tests) ──")
s4_summary = s4_results[["tid", "description", "unsec_breaches", "sec_breaches",
                          "unsec_security", "sec_security"]]
print(s4_summary.to_string(index=False))
print(f"\n  Totals: breaches {s4_unsec_breaches} → {s4_sec_breaches} "
      f"(100% elimination), "
      f"PASS rate: {len(s4_results) - s4_unsec_fail}/{len(s4_results)} → "
      f"{len(s4_results) - s4_sec_fail}/{len(s4_results)}")

  TABLE 2: SECURITY BREACH EVOLUTION

── Step 1: Baseline breach rates (8 queries) ──
      Engine  Total Queries  Total Breaches Breach Rate Security
      chroma              8              33        100%     FAIL
        bm25              8              26         88%     FAIL
ensemble_rrf              8              30        100%     FAIL

── Step 2: Engine comparison (14 queries) ──
      Engine  Tot_Breaches  Queries_FAIL
 chroma_only            53            13
   bm25_only            39            11
baseline_0.5            47            13
     dynamic            46            13

── Step 3: Parent-Child reconstruction effect (8 tests) ──
tid                                 description  before_breaches  after_breaches  merges  drops before_sec after_sec match
 T1              Password — authorized engineer                0               0       1      0       PASS      PASS  PASS
 T2        Password — unauthorized sales intern                3               2       0      1  

In [4]:
# Cell 4 — Table 3: Latency comparison

print("=" * 90)
print("  TABLE 3: LATENCY ANALYSIS (ms)")
print("=" * 90)

latency_table = pd.DataFrame([
    {"Pipeline": "ChromaDB only (Step 1)", "Avg Latency (ms)": 8.3, "Notes": "Vector search, no filtering"},
    {"Pipeline": "BM25 only (Step 1)", "Avg Latency (ms)": 0.2, "Notes": "Pure lexical, no security"},
    {"Pipeline": "Baseline RRF α=0.5 (Step 1)", "Avg Latency (ms)": 8.1, "Notes": "Chroma + BM25, no filtering"},
    {"Pipeline": "Dynamic RRF (Step 2)", "Avg Latency (ms)": 8.3, "Notes": "Intent router + variable α"},
    {"Pipeline": "Reconstruction (Step 3)", "Avg Latency (ms)": 8.4, "Notes": "RRF + parent-child (<0.1ms overhead)"},
    {"Pipeline": "Unsecured baseline (Step 4)", "Avg Latency (ms)": round(s4_results['unsec_lat_ms'].mean(), 1),
     "Notes": "Same as baseline, benchmark measurement"},
    {"Pipeline": "Secure Asymmetric (Step 4)", "Avg Latency (ms)": round(s4_results['sec_lat_ms'].mean(), 1),
     "Notes": "Pre+post filtering, +1.0ms penalty"},
])

print(latency_table.to_string(index=False))

# Step 4 latency breakdown by category
print("\n── Step 4: Latency by user category ──")
for prefix, label in [("S", "Unauthorized (S1-S5)"), ("A", "Authorized (A1-A3)"),
                       ("X", "Cross-dept (X1-X2)"), ("P", "Public (P1-P2)")]:
    sub = s4_results[s4_results["tid"].str.startswith(prefix)]
    if len(sub) == 0:
        continue
    u = sub["unsec_lat_ms"].mean()
    s = sub["sec_lat_ms"].mean()
    d = s - u
    print(f"  {label:30s}  unsec={u:.1f}ms  secure={s:.1f}ms  Δ={d:+.1f}ms ({d/u*100:+.0f}%)")

  TABLE 3: LATENCY ANALYSIS (ms)
                   Pipeline  Avg Latency (ms)                                   Notes
     ChromaDB only (Step 1)               8.3             Vector search, no filtering
         BM25 only (Step 1)               0.2               Pure lexical, no security
Baseline RRF α=0.5 (Step 1)               8.1             Chroma + BM25, no filtering
       Dynamic RRF (Step 2)               8.3              Intent router + variable α
    Reconstruction (Step 3)               8.4    RRF + parent-child (<0.1ms overhead)
Unsecured baseline (Step 4)               9.0 Same as baseline, benchmark measurement
 Secure Asymmetric (Step 4)              10.0      Pre+post filtering, +1.0ms penalty

── Step 4: Latency by user category ──
  Unauthorized (S1-S5)            unsec=9.0ms  secure=9.0ms  Δ=-0.0ms (-0%)
  Authorized (A1-A3)              unsec=8.6ms  secure=11.7ms  Δ=+3.0ms (+35%)
  Cross-dept (X1-X2)              unsec=9.6ms  secure=11.2ms  Δ=+1.6ms (+16%)
  Publi

In [5]:
# Cell 5 — Table 4: Step 2 per-query MRR delta analysis

print("=" * 90)
print("  TABLE 4: STEP 2 — INTENT ROUTER IMPACT (per-query delta)")
print("=" * 90)

print(s2_delta.to_string(index=False))

# Aggregate — all values are strings ("=", "+1", "-0.5000", etc.)
regressed = s2_delta["dMRR"].astype(str).apply(lambda x: x.startswith("-")).sum()
unchanged = s2_delta["dMRR"].astype(str).apply(lambda x: x == "=").sum()
improved = len(s2_delta) - regressed - unchanged

def parse_delta(x):
    s = str(x).strip()
    if s == "=":
        return 0
    try:
        return int(s)
    except ValueError:
        return 0

r5_net = s2_delta["dR@5"].apply(parse_delta).sum()
breach_net = s2_delta["dBreach"].apply(parse_delta).sum()

print(f"\n  MRR: {improved} improved, {regressed} regressed, {unchanged} unchanged")
print(f"  R@5 net change: {r5_net:+d}")
print(f"  Breach net change: {breach_net:+d}")

  TABLE 4: STEP 2 — INTENT ROUTER IMPACT (per-query delta)
Query         Intent  Alpha  MRR_base  MRR_dyn    dMRR dR@3 dR@5 dBreach
   Q1     conceptual    0.8       1.0      1.0       =    =   +1      -1
   Q2     conceptual    0.8       1.0      0.5 -0.5000    =    =       =
   Q3 exact_critical    0.2       1.0      1.0       =    =    =      -1
   Q4 exact_critical    0.2       1.0      1.0       =    =    =      -1
   Q5          exact    0.2       1.0      1.0       =    =   -1       =
   Q6     conceptual    0.8       1.0      1.0       =    =   +1      +1
   Q7          exact    0.2       1.0      1.0       =    =    =      -1
   Q8     conceptual    0.8       1.0      0.5 -0.5000   -1    =       =
   Q9     conceptual    0.8       0.5      0.5       =    =    =      +1
  Q10     conceptual    0.8       1.0      1.0       =    =    =      +1
  Q11          exact    0.2       1.0      1.0       =    =    =       =
  Q12 exact_critical    0.2       1.0      1.0       =    =    = 

In [6]:
# Cell 6 — Table 5: Step 4 MRR by category (authorized vs unauthorized)

print("=" * 90)
print("  TABLE 5: STEP 4 — MRR PRESERVATION FOR AUTHORIZED USERS")
print("=" * 90)

for prefix, label in [("S", "Security tests (unauthorized, should be 0)"),
                       ("A", "Authorized access (should be preserved)"),
                       ("X", "Cross-department (should be 0)"),
                       ("P", "Public queries (should be preserved)")]:
    sub = s4_results[s4_results["tid"].str.startswith(prefix)]
    if len(sub) == 0:
        continue
    print(f"\n  {label}:")
    for _, row in sub.iterrows():
        print(f"    {row['tid']}: MRR {row['unsec_mrr']:.3f} → {row['sec_mrr']:.3f}  "
              f"sec: {row['unsec_security']} → {row['sec_security']}")
    print(f"    Avg MRR: {sub['unsec_mrr'].mean():.3f} → {sub['sec_mrr'].mean():.3f}")

  TABLE 5: STEP 4 — MRR PRESERVATION FOR AUTHORIZED USERS

  Security tests (unauthorized, should be 0):
    S1: MRR 1.000 → 0.000  sec: FAIL → PASS
    S2: MRR 1.000 → 0.000  sec: FAIL → PASS
    S3: MRR 1.000 → 0.000  sec: FAIL → PASS
    S4: MRR 1.000 → 0.000  sec: FAIL → PASS
    S5: MRR 1.000 → 0.000  sec: FAIL → PASS
    Avg MRR: 1.000 → 0.000

  Authorized access (should be preserved):
    A1: MRR 1.000 → 1.000  sec: PASS → PASS
    A2: MRR 1.000 → 1.000  sec: FAIL → PASS
    A3: MRR 1.000 → 1.000  sec: FAIL → PASS
    Avg MRR: 1.000 → 1.000

  Cross-department (should be 0):
    X1: MRR 1.000 → 0.000  sec: FAIL → PASS
    X2: MRR 0.333 → 0.000  sec: FAIL → PASS
    Avg MRR: 0.667 → 0.000

  Public queries (should be preserved):
    P1: MRR 1.000 → 1.000  sec: FAIL → PASS
    P2: MRR 1.000 → 1.000  sec: FAIL → PASS
    Avg MRR: 1.000 → 1.000


In [7]:
# Cell 7 — Export consolidated report

consolidated = {
    "phase": "Phase 2 — Retrieval Strategy Optimization",
    "corpus": {"strategy": "custom_rbac", "total_chunks": 33, "pii_chunks": 15},
    "step1_baseline": {
        "engines": ["ChromaDB", "BM25", "Ensemble RRF"],
        "mrr_ensemble": 0.8929,
        "total_breaches_ensemble": 47,
        "security_pass_rate": "0/14",
        "avg_latency_ms": 8.1,
        "finding": "BM25 leaks PII — no native metadata filtering"
    },
    "step2_dynamic_rrf": {
        "mrr_baseline": 0.8929,
        "mrr_dynamic": 0.8214,
        "mrr_delta": -0.0715,
        "r5_delta": "+1 net",
        "breach_delta": "-1 net",
        "finding": "Marginal impact due to structural ceiling (corpus homogeneity, size ratio, engine overlap)"
    },
    "step3_parent_child": {
        "parent_index_size": 15,
        "test_accuracy": "8/8 (100%)",
        "breach_reduction": "19 → 14 (-26%)",
        "security_pass_rate": "1/8 → 2/8",
        "reconstruction_latency_ms": "<0.1",
        "context_expansion": "4x to 22x (21-101ch → 152-462ch)",
        "finding": "Context restored for authorized users; residual breaches from non-PII restricted chunks"
    },
    "step4_asymmetric_fusion": {
        "total_breaches": "38 → 0 (100% elimination)",
        "security_pass_rate": "1/12 → 12/12 (100%)",
        "latency_penalty_ms": "+1.03 (+11%)",
        "mrr_authorized": "1.000 (preserved)",
        "mrr_unauthorized": "0.000 (correct by design)",
        "finding": "Zero-Trust achieved — pre-filtering ChromaDB + post-filtering BM25"
    },
    "conclusions": {
        "retrieval_quality": "MRR 0.893 baseline — ranking quality is not the bottleneck",
        "context_restoration": "Parent-child reconstruction expands PII fragments 4-22x with RBAC gate",
        "security": "Asymmetric fusion eliminates 100% of breaches with +1ms latency penalty",
        "architecture": "Intent Router → Secure Asymmetric RRF → Parent-Child Reconstruction"
    }
}

json_path = os.path.join(RES, "ph2_step5_consolidated_report.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(consolidated, f, indent=2, ensure_ascii=False)
print(f"Exported: {json_path}")

# Also export the master table
csv_path = os.path.join(RES, "ph2_step5_master_comparison.csv")
master.to_csv(csv_path, index=False)
print(f"Exported: {csv_path}")

print("\n" + "=" * 90)
print("  PHASE 2 COMPLETE — All 5 steps executed successfully")
print("=" * 90)
print("""
  Key outcomes:
  • Retrieval quality: MRR = 0.893 (baseline already strong)
  • Context restoration: PII fragments expanded 4-22x via parent-child merge
  • Security: 100% breach elimination via asymmetric pre+post filtering
  • Latency penalty: +1.0ms (negligible vs LLM inference ~500-2000ms)
  • Architecture validated: Intent Router + Secure Asymmetric RRF + Parent-Child
""")

Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph2-retrieval-strategy\..\..\data\results\notebook_results\ph2\ph2_step5_consolidated_report.json
Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph2-retrieval-strategy\..\..\data\results\notebook_results\ph2\ph2_step5_master_comparison.csv

  PHASE 2 COMPLETE — All 5 steps executed successfully

  Key outcomes:
  • Retrieval quality: MRR = 0.893 (baseline already strong)
  • Context restoration: PII fragments expanded 4-22x via parent-child merge
  • Security: 100% breach elimination via asymmetric pre+post filtering
  • Latency penalty: +1.0ms (negligible vs LLM inference ~500-2000ms)
  • Architecture validated: Intent Router + Secure Asymmetric RRF + Parent-Child

